# The control Class

In this notebook we introduce the control class.

The software control is a library for solving certain PDE-constrained optimization problems. The software employs the Firedrake system to derive the finite element discretization of the problems considered, using the Python interface to PETSc for the derivation of the KKT conditions and the definition of the linear solvers.


## A basic example: a linear Poisson control problem
Given $\beta > 0$, a force function $f$, and a desired state $v_d$, we consider the following linear Poisson control problem:

$$
\min_{v,u} \frac{1}{2} \| v - v_d \| ^2 + \frac{1}{\beta} \| u \| ^2$$

subject to:

$$
    -\nabla^2 v = f + u \qquad \mathrm{in} \; \Omega := (-1, 1)^2, 
$$

with $v = 1$ on $\partial \Omega$. For the problem considered here, we set $v_d = \cos(\frac{\pi x_1}{2}) \cos(\frac{\pi x_2}{2}) + 1$ and $f = \frac{\pi^2}{2} \cos(\frac{\pi x_1}{2}) \cos(\frac{\pi x_2}{2})$, with analytical state solution given by $v = v_d$. Further, the adjoint variable is $\zeta = 0$.

We consider a uniform grid with 10 points in each spatial direction, and employ $P_1$ finite elements as the spatial discretization.

In order to define the problem, we first import the Stationary module from the control class.

In [ ]:
from firedrake import *
from control.control import Stationary

We now defined the geometry of the problem and the function space where we seek the solution.


In [ ]:
mesh = RectangleMesh(10, 10, 1.0, 1.0, originX=-1.0, originY=-1.0)
space_0 = FunctionSpace(mesh, "Lagrange", 1)

Then, we define the desired state $v_d$, the force function $f$, and the boundary conditions to apply to $v$.

In [ ]:
# the desired state
def desired_state(test):
    space = test.function_space()
    mesh = space.mesh()
    X = SpatialCoordinate(mesh)
    x_1 = X[0]
    x_2 = X[1]

    v_d = Function(space, name="v_d")
    v_d.interpolate(cos(0.5 * pi * x_1) * cos(0.5 * pi * x_2) + 1.0)

    return inner(v_d, test) * dx, v_d


# the force function
def force_f(test):
    space = test.function_space()
    mesh = space.mesh()
    X = SpatialCoordinate(mesh)
    x_1 = X[0]
    x_2 = X[1]

    f = Function(space, name="f")
    f.interpolate(0.5 * pi * pi * cos(0.5 * pi * x_1) * cos(0.5 * pi * x_2))

    return inner(f, test) * dx


# the boundary conditions
bcs_v = DirichletBC(space_0, 1.0, "on_boundary")

Finally, we define the form that represents the differential operator in space.

In [ ]:
# the forward form
def forw_diff_operator(trial, test, v):
    return inner(grad(trial), grad(test)) * dx

Now, we can define the Poisson control problem to be solved. This is done by passing the space where to find the solution, the bilinear form, the desired state, the force function, the value of $\beta$, and the boundary conditions to the module Stationary. For this example, we set $\beta=10^{-3}$. Note that the default option for $\beta$ is $10^{-3}$, so the kwarg beta can be avoided in the following call.

In [ ]:
Poisson_control = Stationary(space_0, forw_diff_operator, desired_state=desired_state,
                             force_function=force_f, beta=1.0e-3, bcs_v=bcs_v)

In order to solve the control problem defined above, we have to call the module linear_solve(). The inner-built solver is preconditioned GMRES, restarted every 10 iterations. The solver runs up to a reduction of $10^{-6}$ on the absolute or relative residual is achieved. In case one wishes to change the settings, one has to pass a dictionary containing the settings to the solver. For example, suppose we want to run full FGMRES up to a tolerance of $10^{-8}$ on the absolute or relative residual, with the story of the residual printed on output. Then, we have to pass the following dictionary to the solver.

In [ ]:
solver_parameters = {"linear_solver": "gmres",
                     "maximum_iterations": 50,
                     "relative_tolerance": 1.0e-8,
                     "absolute_tolerance": 1.0e-8,
                     "monitor_convergence": True}

The software employs as default preconditioner a block-triangular approximation based on the so called matching-strategy. The $(1,1)$-block is approximated with a Jacobi iteration, and each block of the Schur complement approximation is inverted inexactly with 2 cycles of an algebraic multigrid. The user can modify the setting by passing the kwargs auxiliary_sp, as follows.

For example, suppose we want to employ a fixed number of Chebyshev semi-iterations with Jacobi splitting as solver for the $(1,1)$-block. We first need to create a dictionary containing the informations of the solver for the Chebyshev semi-iteration.

In [ ]:
# bounds on the eigenvalues for the preconditioned mass matrix
e_min_p = 0.5
e_max_p = 2.0

sp_11block = {"ksp_type": "chebyshev",
              "pc_type": "jacobi",
              "ksp_chebyshev_eigenvalues": f"{e_min_p:.16e}, {e_max_p:.16e}",
              "ksp_chebyshev_esteig": "0.0,0.0,0.0,0.0",
              "ksp_chebyshev_esteig_steps": 0,
              "ksp_chebyshev_esteig_noisy": False,
              "ksp_max_it": 20,
              "ksp_atol": 0.0,
              "ksp_rtol": 0.0}

Next, we have to create a dictionary to pass to the solver.

In [ ]:
auxiliary_sp = {"sp_11block": sp_11block}

Finally, we can call the linear solver as follows.

In [ ]:
Poisson_control.linear_solve(
        solver_parameters=solver_parameters, auxiliary_sp=auxiliary_sp,
        print_error=False, outputs=False, plots=False)

The state and adjoint solutions are then stored in the variables Poisson_control._v and Poisson_control._zeta, respectively.

The package allows for printing the the $L^2$-discrepancy between the desired state $v_d$ and the numerical solution by calling print_error=True in the call above (the default option is True). Further, .pvd and .h5 outputs containing the state and the adjoint solutions; this is done by passing outputs=True to the call (the default option is True). Finally, plots of the solution can be obtained by setting plots=True (the default option is False).

We can now postprocess the solutions obtained, checking the numerical errors.

In [ ]:
v = Poisson_control._v
zeta = Poisson_control._zeta

true_v = Function(space_0)
true_zeta = Function(space_0)

# evaluating the true solution
X = SpatialCoordinate(mesh)
x_1 = X[0]
x_2 = X[1]

true_v.interpolate(cos(0.5 * pi * x_1) * cos(0.5 * pi * x_2) + 1.0)
true_zeta.zero()

# evaluating the norm of the error
v_error = np.sqrt(abs(assemble(
    inner(v - true_v, v - true_v) * dx)))
zeta_error = np.sqrt(abs(assemble(
    inner(zeta - true_zeta, zeta - true_zeta) * dx)))

print(f"{v_error=}")
print(f"{zeta_error=}")